<a href="https://colab.research.google.com/github/nishanthyd/Monkey-Chunky/blob/master/AI_Summit_TripMate_Workshop.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TripMate on Google Colab

This notebook runs your TripMate voice agent. The shared web page only connects a browser to whichever LiveKit project you paste in.

 Each student adds their own API keys in the Colab Secrets panel.

---


## Before you run anything

1. Create a free project at [LiveKit Cloud](https://cloud.livekit.io).
2. Turn **Development token server** on (Settings → Keys).
3. Add Colab Secrets (**Notebook access** ON):

---


## Workshop — five stages

Say this every stage:

> Plan a 4-day Goa trip with beaches and nightlife, medium budget. What hotel, and what price per night?

### How to run a stage

1. Run the setup cells once (install, secrets, catalog, model download, runner).
2. Run **one** stage cell. Leave it running — a spinning cell means the agent is online.
3. Wait until the cell prints `registered worker`.
4. Connect on the website and say the sentence above.
5. **Stop that cell** (⬛) before you run the next stage. The old worker keeps the old code until you stop it.



### 🚀 The Interactive Roadmap

> *   **Stage 1:** We build a simple voice chatbot. It speaks, but it happily lies about the prices because it has no tools.
> *   **Stage 2:** We try to fix it using instructions only (*'Do not lie'*)—and we'll see the limits of prompting.
> *   **Stage 3:** We give it its first 'hand'—a tool to fetch real hotel data from our Excel sheet.
> *   **Stage 4:** We give it all three tools and an orchestrated planning order.
> *   **Stage 5:** We add high-end conversational turn detection so it knows when to wait for you to finish a thought instead of cutting you off.


## Setup — install

This installs the voice stack once. Later stages reuse it. The turn-detection model is only used in stage 5, but we install it now so that stage does not stop for a download.


In [1]:
# Install agent packages with flexible modern versions to prevent dependency conflicts
!pip install -q -U livekit-agents livekit-plugins-silero livekit-plugins-simli livekit-plugins-turn-detector livekit-plugins-noise-cancellation python-dotenv pandas openpyxl

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 5.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.6/44.6 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 66.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.3/51.3 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.8/11.8 MB 96.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 85.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.6/73.6 MB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 76.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.0/35.0 MB 44.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.3/185.3 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.9/34.9 MB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 151.3/151.3 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8

## Setup — your LiveKit secrets

Click the **key** icon on the left. Add these secrets and turn **Notebook access** on. Paste the values **without quotes**.

| Secret name | What to paste |
| --- | --- |
| `LIVEKIT_URL` | Project URL, `wss://your-project.livekit.cloud` |
| `LIVEKIT_API_KEY` | API key |
| `LIVEKIT_API_SECRET` | API secret |

Do not paste the token server ID here. That ID goes in the website, not in this notebook.

Speech, the language model, and the voice are billed to **your** LiveKit project. You do not need separate OpenAI, Deepgram, or Cartesia keys.


In [ ]:
import os
from pathlib import Path

try:
    from google.colab import userdata
except ImportError as exc:
    raise SystemExit(
        "Open this notebook in Google Colab. Keys are read from the Colab Secrets panel."
    ) from exc


def secret(name: str) -> str:
    try:
        value = userdata.get(name)
    except Exception as exc:
        raise SystemExit(
            f"Could not read Colab secret {name}. Add it in the key panel and turn Notebook access on. "
            "Do not paste the value into a cell."
        ) from exc
    value = (value or "").strip().strip('"').strip("'")
    if not value:
        raise SystemExit(f"Colab secret {name} is empty.")
    return value


url = secret("LIVEKIT_URL").rstrip("/")
if url.startswith("https://"):
    url = "wss://" + url[len("https://"):]
elif url.startswith("http://"):
    url = "ws://" + url[len("http://"):]
elif "://" not in url:
    url = "wss://" + url
if "sandbox.livekit.io" in url or not url.endswith(".livekit.cloud"):
    raise SystemExit(
        "LIVEKIT_URL must be the project URL, like wss://your-project.livekit.cloud. "
        "Not the token server ID, and not the sandbox.livekit.io URL."
    )

os.environ["LIVEKIT_URL"] = url
os.environ["LIVEKIT_API_KEY"] = secret("LIVEKIT_API_KEY")
os.environ["LIVEKIT_API_SECRET"] = secret("LIVEKIT_API_SECRET")
print("Using LiveKit project:", url.split("://", 1)[-1])
print("API key and secret are loaded. Values are not printed.")


Using LiveKit project: test-agent-5f42hkx0.livekit.cloud
API key and secret are loaded. Values are not printed.


## Setup — the catalog


Stages 1 and 2 do not read this file. They are allowed to invent.

Stages 3, 4, and 5 may speak a hotel or a city **only if it is in this workbook**. Upload `tripmate_catalog.xlsx` when the next cell asks. It has two sheets we will use:

- `destinations` — city, interests, score, why
- `hotels` — city, name, price per night, rating, budget

If a traveler asks for a city that is not a row, the tool returns nothing. The agent should say so. That is the lesson: the model does not get to fill the gap.


In [ ]:
from pathlib import Path
import pandas as pd

path = Path("/content/Travel_catalog.xlsx")
if not path.exists():
    path = Path("/content/tripmate_catalog.xlsx")

try:
    from google.colab import files
except ImportError:
    files = None

if not path.exists():
    if files is None:
        raise SystemExit("Put tripmate_catalog.xlsx next to this notebook, then re-run this cell.")
    print("Upload tripmate_catalog.xlsx")
    uploaded = files.upload()
    if not uploaded:
        raise SystemExit("No file uploaded.")
    path = Path("/content/tripmate_catalog.xlsx")
    path.write_bytes(next(iter(uploaded.values())))


def _split_list(value):
    # Excel cells store "beaches, nightlife" as one string. Tools want a list.
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return []
    text = str(value).replace("|", ",").replace(";", ",")
    return [part.strip() for part in text.split(",") if part.strip()]


def _city_key(city: str) -> str:
    return city.strip().lower()


destinations_df = pd.read_excel(path, sheet_name="destinations")
hotels_df = pd.read_excel(path, sheet_name="hotels")
activities_df = pd.read_excel(path, sheet_name="activities")

DESTINATIONS = []
for _, row in destinations_df.iterrows():
    DESTINATIONS.append({
        "city": str(row["city"]).strip(),
        "interests": [item.lower() for item in _split_list(row.get("interests"))],
        "score": float(row["score"]),
        "why": str(row["why"]).strip(),
    })

HOTELS = {}
for _, row in hotels_df.iterrows():
    HOTELS.setdefault(_city_key(str(row["city"])), []).append({
        "name": str(row["name"]).strip(),
        "price_per_night": int(row["price_per_night"]),
        "rating": float(row["rating"]),
        "budget": str(row["budget"]).strip().lower(),
    })

ACTIVITIES = {}
for _, row in activities_df.iterrows():
    ACTIVITIES.setdefault(_city_key(str(row["city"])), []).append({
        "day": int(row["day"]),
        "items": _split_list(row.get("items")),
        "interests": [item.lower() for item in _split_list(row.get("interests"))],
    })

AVAILABLE_CITIES = sorted(place["city"] for place in DESTINATIONS)
print("Catalog:", path.resolve())
print("Cities:", ", ".join(AVAILABLE_CITIES))
print(f"Loaded: {len(DESTINATIONS)} destinations, {sum(len(v) for v in HOTELS.values())} hotels, {sum(len(v) for v in ACTIVITIES.values())} activity plans")


Catalog: /content/Travel_catalog.xlsx
Cities: Andaman, Goa, Gokarna, Jaipur, Manali, Munnar, Mussoorie, Pondicherry, Rishikesh, Udaipur
Loaded: 10 destinations, 22 hotels, 23 activity plans


## Setup — turn-detection files


Stage 5 needs a small model file (`languages.json`) so the agent can tell when you have finished speaking. Download it now. Stages 1–4 do not use it.


In [ ]:
!python -m livekit.agents download-files
print("Model files ready.")


discovered 4 plugin package(s): livekit.plugins.noise_cancellation, livekit.plugins.silero, livekit.plugins.simli, livekit.plugins.turn_detector
downloading files for livekit.plugins.silero
finished downloading files for livekit.plugins.silero
downloading files for livekit.plugins.simli
finished downloading files for livekit.plugins.simli
downloading files for livekit.plugins.turn_detector
NumExpr defaulting to 2 threads.
HTTP Request: GET https://huggingface.co/api/agent-harnesses "HTTP/1.1 200 OK"
HTTP Request: GET https://huggingface.co/api/models/livekit/turn-detector/revision/v1.2.2-en "HTTP/1.1 200 OK"
HTTP Request: HEAD https://huggingface.co/livekit/turn-detector/resolve/ebcab0c09c2b62d926e92180d364df3aaae68a09/config.json "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/livekit/turn-detector/ebcab0c09c2b62d926e92180d364df3aaae68a09/config.json?%2Flivekit%2Fturn-detector%2Fresolve%2Febcab0c09c2b62d926e92180d364df3aaae68a09%2Fc

## Setup — how a stage cell stays running

Every stage cell ends by calling `run_stage(server)`.

That call does not return. The cell keeps spinning because the worker is online, waiting for someone to join a room. **Spinning is success.** Stopping the cell hangs up the agent.

### What happens when you click Connect

```
Browser clicks Connect
        │
        ▼
LiveKit Cloud creates a Room
        │
        ▼
Cloud checks: "Any registered worker?"
        │  (your spinning cell registered one)
        ▼
Cloud dispatches a Job to your agent
        │
        ▼
Your @server.rtc_session() function runs
        │
        ▼
AgentSession.start() → agent joins the Room
        │
        ▼
Agent is now a participant, same as the user
```

### The voice pipeline inside the agent

```
Mic audio (WebRTC)
   → VAD (Silero, local, ~2 MB)      — speech or silence?
   → STT (Deepgram, cloud API)       — speech → text
   → LLM (GPT-4.1-mini, cloud API)   — text → response
   → TTS (Cartesia, cloud API)       — response → speech
   → Published back via WebRTC        — user hears the agent
```

VAD is the **only model that runs locally**. Everything else is a cloud API call — that is why you need LiveKit keys but not a separate VAD key.

### Interruption

When you talk over the agent: VAD detects your voice → agent stops TTS mid-word → flushes the LLM stream → STT starts on your new speech → pipeline restarts. Under 300 ms.

### Colab detail

`JobExecutorType.THREAD` keeps the agent inside this notebook. LiveKit's default is a separate process, and a function written in a Colab cell cannot be sent to that process.

If you forget to stop the previous stage, `run_stage` closes it before starting the new one.


In [ ]:
import logging

from livekit.agents import JobExecutorType

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(message)s",
    force=True,
)

_current_server = None


async def run_stage(new_server, stage_name: str):
    """Start this stage and leave the cell running until you press stop."""
    global _current_server
    if _current_server is not None and not _current_server._closed:
        print("Closing the previous stage...")
        await _current_server.aclose()
    _current_server = new_server
    print(f"{stage_name} is starting.")
    print("Wait for 'registered worker', then connect on the website and say the Goa sentence.")
    print("Stop this cell before you run the next stage.\n")
    await new_server.run(devmode=True)  ## the agent registers itself with LiveKit Cloud —
    # it says 'I am online, send me work' — and then blocks forever, heartbeating every few seconds.
    # It is a server, waiting for you instructions


# Stage 1 — voice chatbot

**Change:** none. This is the smallest agent. It can hear, think, and speak.

Look at the instruction. It is one sentence: "You are a helpful voice AI assistant." There is no tool and no spreadsheet. The language model will answer from whatever it already knows, including prices it is happy to invent.

The pipeline:

| Step | Plugin | Runs where | What it does |
| --- | --- | --- | --- |
| VAD | `silero.VAD.load()` | **Locally** (2 MB model in your Colab) | Decides if audio is speech or silence |
| STT | `deepgram/nova-3` | Cloud API | Converts speech → text |
| LLM | `openai/gpt-4.1-mini` | Cloud API | Generates response text |
| TTS | `cartesia/sonic-2` | Cloud API | Converts text → speech |

**Say:** "Plan a 4-day Goa trip with beaches and nightlife, medium budget. What hotel, and what price per night?"

**Listen for:** a fluent answer and a price that is **not** in the Excel sheet. There will be no `TOOL` line in this cell.

That price is the thing we are about to take away.


In [ ]:
import logging

from livekit.agents import (
    Agent,
    AgentServer,
    AgentSession,
    JobContext,
    JobExecutorType,
    RoomInputOptions,
)
from livekit.plugins import noise_cancellation, silero ##

# The entire personality is this string. Nothing else constrains the facts.
class Assistant(Agent):
    def __init__(self) -> None:
        super().__init__(instructions="You are a helpful voice AI assistant.")


server = AgentServer(job_executor_type=JobExecutorType.THREAD)

# When someone clicks Connect on the website, LiveKit Cloud creates a Room.
# Cloud then looks at the registered workers and dispatches a Job to your agent —
# that is the @server.rtc_session() decorator. Your entrypoint function runs,
# creates an AgentSession, and joins the room as a second participant alongside the browser user.
@server.rtc_session()
async def entrypoint(ctx: JobContext):
    # Ears, brain, voice. This pipeline stays the same through stage 4.
    session = AgentSession(
        stt="deepgram/nova-3",
        llm="openai/gpt-4.1-mini",
        tts="cartesia/sonic-2",
        vad=silero.VAD.load(),
    )
    await session.start(
        agent=Assistant(),
        room=ctx.room,
        room_input_options=RoomInputOptions(
            noise_cancellation=noise_cancellation.BVC(),
        ),
    )
    await ctx.connect()


await run_stage(server, "Stage 1 — voice chatbot")


2026-10-21 14:32:52,130 INFO shutting down worker
2026-10-21 14:32:52,135 INFO process exiting
2026-10-21 14:32:52,269 INFO starting worker
2026-10-21 14:32:52,270 INFO plugin registered
2026-10-21 14:32:52,271 INFO plugin registered
2026-10-21 14:32:52,273 INFO plugin registered
2026-10-21 14:32:52,275 INFO plugin registered
2026-10-21 14:32:52,276 INFO preloading plugins
2026-10-21 14:32:52,277 INFO starting inference executor


Closing the previous stage...
Stage 1 — voice chatbot is starting.
Wait for 'registered worker', then connect on the website and say the Goa sentence.
Stop this cell before you run the next stage.



2026-10-21 14:32:52,291 INFO initializing process
2026-10-21 14:33:05,377 INFO process initialized
2026-10-21 14:33:05,383 INFO HTTP server listening on :46017
2026-10-21 14:33:05,420 INFO registered worker
2026-10-21 14:33:17,050 INFO received job request
2026-10-21 14:33:17,054 WARNING no warmed process available for job, waiting for one to be created
2026-10-21 14:33:17,058 INFO initializing job runner
2026-10-21 14:33:17,063 INFO job runner initialized
2026-10-21 14:33:17,166 WARNING RoomInputOptions and RoomOutputOptions are deprecated, use RoomOptions instead
2026-10-21 14:33:17,179 INFO adaptive interruption is disabled by default in production mode
2026-10-21 14:33:17,186 WARNING event loop blocked for 107ms at "/usr/lib/python3.13/threading.py", line 363, in wait; synchronous work on the agent loop delays audio and turn handling, move it to a thread or an async client
2026-10-21 14:33:17,227 INFO livekit_signaling::signal_stream:53:livekit_signaling::signal_stream - connecting

CancelledError: 

# Stage 2 — new instructions only


Stop stage 1 first.

**Change:** the instruction string. The pipeline is the same. There is still no tool.

The new instruction gives the agent a name and forbids invented prices. Read it out loud before you run the cell. Notice that we did not give it a hotel list. We only asked it to be honest.

**Say the same Goa sentence.**

**Listen for:** it introduces itself as TripMate. When you ask for a price, it should admit it does not know, or it will still slip and invent one. Either way, nothing in this cell prints `TOOL`, because there is nothing to call.

This is the limit of a prompt. Manners changed. Knowledge did not.


In [ ]:
from livekit.agents import (
    Agent,
    AgentServer,
    AgentSession,
    JobContext,
    JobExecutorType,
    RoomInputOptions,
)
from livekit.plugins import noise_cancellation, silero


class TripMate(Agent):
    def __init__(self) -> None:
        super().__init__(
            instructions=(
                "You are TripMate, a voice travel planner for trips in India. "
                "Speak in short spoken sentences. "
                "If you do not have a real hotel name or a real price, say you do not know. "
                "Do not invent prices."
            ),
        )


server = AgentServer(job_executor_type=JobExecutorType.THREAD)


@server.rtc_session()
async def entrypoint(ctx: JobContext):
    session = AgentSession(
        stt="deepgram/nova-3",
        llm="openai/gpt-4.1-mini",
        tts="cartesia/sonic-2:9626c31c-bec5-4cca-baa8-f8ba9e84c8bc",
        vad=silero.VAD.load(),
    )
    await session.start(
        agent=TripMate(),
        room=ctx.room,
        room_input_options=RoomInputOptions(
            noise_cancellation=noise_cancellation.BVC(),
        ),
    )
    await ctx.connect()


await run_stage(server, "Stage 2 — instructions only")


2026-10-21 13:43:11,047 INFO shutting down worker
2026-10-21 13:43:11,052 INFO starting worker
2026-10-21 13:43:11,053 INFO plugin registered
2026-10-21 13:43:11,054 INFO preloading plugins
2026-10-21 13:43:11,057 INFO HTTP server listening on :37565
2026-10-21 13:43:11,088 INFO registered worker


Closing the previous stage...
Stage 2 — instructions only is starting.
Wait for 'registered worker', then connect on the website and say the Goa sentence.
Stop this cell before you run the next stage.



2026-10-21 13:43:27,972 INFO received job request
2026-10-21 13:43:27,978 WARNING no warmed process available for job, waiting for one to be created
2026-10-21 13:43:27,982 INFO initializing job runner
2026-10-21 13:43:27,985 INFO job runner initialized
2026-10-21 13:43:28,069 WARNING RoomInputOptions and RoomOutputOptions are deprecated, use RoomOptions instead
2026-10-21 13:43:28,082 INFO adaptive interruption is disabled by default in production mode
2026-10-21 13:43:28,132 INFO livekit_signaling::signal_stream:53:livekit_signaling::signal_stream - connecting to wss://test-agent-5f42hkx0.oashburn1b.production.livekit.cloud/rtc/v1
2026-10-21 13:43:28,185 INFO HTTP Request: GET https://agent-gateway.livekit.cloud/v1/models "HTTP/1.1 200 OK"
2026-10-21 13:43:29,113 INFO RoomIO linked to participant
2026-10-21 13:43:32,865 INFO HTTP Request: POST https://agent-gateway.livekit.cloud/v1/chat/completions "HTTP/1.1 200 OK"
2026-10-21 13:43:33,968 INFO HTTP Request: POST https://agent-gatewa

CancelledError: 

# Stage 3 — one tool


Stop stage 2 first.

**Change:** one Python function, `find_hotels`, marked with `@function_tool`. The pipeline is still the same.

Three things make it a tool the model can call:

1. The decorator `@function_tool`.
2. The docstring. That text is what the model reads to decide when to call it.
3. The `Args` lines. Those become the arguments the model must fill in.

The function does not ask the language model for a price. It looks up `HOTELS`, which was loaded from the Excel sheet. A `print` runs when the tool is called, so you can see the moment the chatbot becomes a tool-using agent.

The instruction now says: call `find_hotels`, and speak only what it returns. For a medium budget, prefer the cheaper well-rated hotel. In the sheet that is Sea Breeze in Goa, ₹2700 a night, not Sunset Bay.

**Say the same Goa sentence.**

**Listen for:** a line in this cell that starts with `TOOL find_hotels`, then a spoken price that matches the sheet.

Then ask: "What about a hotel in Shimla?" Shimla is not in the sheet. The tool returns no hotels. TripMate should say so instead of inventing one.


In [ ]:
from livekit.agents import (
    Agent,
    AgentServer,
    AgentSession,
    JobContext,
    JobExecutorType,
    RoomInputOptions,
    RunContext,
    function_tool,
)
from livekit.plugins import noise_cancellation, silero


class TripMate(Agent):
    def __init__(self) -> None:
        super().__init__(
            instructions=(
                "You are TripMate, a voice travel planner for trips in India. "
                "Speak in short spoken sentences. "
                "When the traveler asks about a hotel or a price, call find_hotels. "
                "Never invent a hotel name or a price. Speak only what find_hotels returns. "
                "For a medium budget, prefer the lower-priced well-rated hotel, not the most expensive one. "
                "If the tool returns no hotels, say that city is not in your catalog."
            ),
        )

    @function_tool
    async def find_hotels(
        self,
        context: RunContext,
        city: str,
        nights: int,
        budget: str = "medium",
    ) -> dict:
        """Find hotels in a city from the workshop spreadsheet.

        Args:
            city: Destination city, such as Goa.
            nights: Number of nights to stay.
            budget: low, medium, or high.
        """
        # This print is the moment the audience can see a tool call happen.
        print(f"TOOL find_hotels city={city} nights={nights} budget={budget}")
        stays = HOTELS.get(_city_key(city), [])
        if not stays:
            return {
                "city": city,
                "hotels": [],
                "note": f"No hotels in the catalog for {city}.",
                "available_cities": AVAILABLE_CITIES,
            }
        # Medium also returns the high option so the model can compare and reject it.
        allowed = {budget.lower()}
        if budget.lower() == "medium":
            allowed.add("high")
        chosen = [hotel for hotel in stays if hotel["budget"] in allowed] or stays
        return {
            "city": city,
            "nights": nights,
            "hotels": [
                {
                    "name": hotel["name"],
                    "price_per_night_inr": hotel["price_per_night"],
                    "stay_total_inr": hotel["price_per_night"] * nights,
                    "rating": hotel["rating"],
                }
                for hotel in chosen
            ],
        }


server = AgentServer(job_executor_type=JobExecutorType.THREAD)


@server.rtc_session()
async def entrypoint(ctx: JobContext):
    session = AgentSession(
        stt="deepgram/nova-3",
        llm="openai/gpt-4.1-mini",
        tts="cartesia/sonic-2:9626c31c-bec5-4cca-baa8-f8ba9e84c8bc",
        vad=silero.VAD.load(),
    )
    await session.start(
        agent=TripMate(),
        room=ctx.room,
        room_input_options=RoomInputOptions(
            noise_cancellation=noise_cancellation.BVC(),
        ),
    )
    await ctx.connect()


await run_stage(server, "Stage 3 — one tool")


2026-10-21 13:45:47,429 INFO shutting down worker
2026-10-21 13:45:47,433 INFO starting worker
2026-10-21 13:45:47,434 INFO plugin registered
2026-10-21 13:45:47,436 INFO preloading plugins
2026-10-21 13:45:47,441 INFO HTTP server listening on :44419
2026-10-21 13:45:47,474 INFO registered worker


Closing the previous stage...
Stage 3 — one tool is starting.
Wait for 'registered worker', then connect on the website and say the Goa sentence.
Stop this cell before you run the next stage.



2026-10-21 13:46:03,974 INFO received job request
2026-10-21 13:46:03,979 WARNING no warmed process available for job, waiting for one to be created
2026-10-21 13:46:03,981 INFO initializing job runner
2026-10-21 13:46:03,983 INFO job runner initialized
2026-10-21 13:46:04,089 WARNING RoomInputOptions and RoomOutputOptions are deprecated, use RoomOptions instead
2026-10-21 13:46:04,100 INFO adaptive interruption is disabled by default in production mode
2026-10-21 13:46:04,150 INFO livekit_signaling::signal_stream:53:livekit_signaling::signal_stream - connecting to wss://test-agent-5f42hkx0.oashburn1b.production.livekit.cloud/rtc/v1
2026-10-21 13:46:04,204 INFO HTTP Request: GET https://agent-gateway.livekit.cloud/v1/models "HTTP/1.1 200 OK"
2026-10-21 13:46:04,998 INFO RoomIO linked to participant
2026-10-21 13:46:15,269 INFO HTTP Request: POST https://agent-gateway.livekit.cloud/v1/chat/completions "HTTP/1.1 200 OK"
2026-10-21 13:46:16,063 INFO HTTP Request: POST https://agent-gatewa

TOOL find_hotels city=Goa nights=2 budget=medium


2026-10-21 13:46:22,993 INFO HTTP Request: POST https://agent-gateway.livekit.cloud/v1/chat/completions "HTTP/1.1 200 OK"
2026-10-21 13:48:00,430 INFO HTTP Request: POST https://agent-gateway.livekit.cloud/v1/chat/completions "HTTP/1.1 200 OK"


TOOL find_hotels city=Shimla nights=2 budget=medium


2026-10-21 13:48:00,950 INFO HTTP Request: POST https://agent-gateway.livekit.cloud/v1/chat/completions "HTTP/1.1 200 OK"
2026-10-21 13:48:24,997 INFO closing agent session due to participant disconnect (disable via `RoomInputOptions.close_on_disconnect=False`)


CancelledError: 

# Stage 4 — all three tools, and an order


Stop stage 3 first.

**Change:** `search_destinations` and `find_activities`, plus instructions that set the order: search first, then hotels, then activities.

`find_hotels` is unchanged. The new tools:
- `search_destinations` — ranks cities from the destinations sheet by traveler interests
- `find_activities` — builds a day-by-day plan from the activities sheet

The model still decides when to call. The instructions tell it the order. You should see three tool lines:

1. `TOOL search_destinations`
2. `TOOL find_hotels`
3. `TOOL find_activities`

**Say the same Goa sentence.** Beaches and nightlife should put Goa first (score 9.4 in the sheet). The hotel line should still be a sheet price. And now you get a day-by-day plan.

This is a tool-using agent with a plan. It is not more conversational yet. If you pause in the middle of the sentence, it may still jump in. That is stage 5.


In [ ]:
from livekit.agents import (
    Agent,
    AgentServer,
    AgentSession,
    JobContext,
    JobExecutorType,
    RoomInputOptions,
    RunContext,
    function_tool,
)
from livekit.plugins import noise_cancellation, silero


def _normalize_interests(interests: list[str]) -> list[str]:
    cleaned = []
    for interest in interests:
        value = interest.strip().lower()
        if value in {"beach", "beachside"}:
            value = "beaches"
        if value in {"night life", "party", "parties", "clubs"}:
            value = "nightlife"
        if value in {"hill", "hills", "hill station"}:
            value = "mountains"
        if value:
            cleaned.append(value)
    return cleaned


class TripMate(Agent):
    def __init__(self) -> None:
        super().__init__(
            instructions=(
                "You are TripMate, a voice travel planner for trips in India. "
                "Speak in short spoken sentences. "
                "Never invent a city, hotel, price, or activity. "
                "Call search_destinations first when the traveler describes a trip. "
                "After you name a city, call find_hotels for that city, then call find_activities. "
                "For a medium budget, prefer the lower-priced well-rated hotel. "
                "If a tool says the city is not in the catalog, say that and name a city that is."
            ),
        )

    @function_tool
    async def search_destinations(
        self,
        context: RunContext,
        interests: list[str],
        duration: int,
        budget: str = "medium",
    ) -> dict:
        """Rank cities from the workshop spreadsheet for this trip.

        Args:
            interests: What the traveler wants, such as beaches or nightlife.
            duration: Trip length in days.
            budget: low, medium, or high.
        """
        print(f"TOOL search_destinations interests={interests} duration={duration} budget={budget}")
        wanted = set(_normalize_interests(interests))
        ranked = []
        for place in DESTINATIONS:
            overlap = wanted & set(place["interests"])
            if wanted and not overlap:
                continue
            ranked.append({
                "city": place["city"],
                "score": place["score"],
                "matched_interests": sorted(overlap),
                "why": place["why"],
                "duration_days": duration,
                "budget": budget,
            })
        ranked.sort(key=lambda item: item["score"], reverse=True)
        return {"destinations": ranked[:3], "available_cities": AVAILABLE_CITIES}

    @function_tool
    async def find_hotels(
        self,
        context: RunContext,
        city: str,
        nights: int,
        budget: str = "medium",
    ) -> dict:
        """Find hotels in a city from the workshop spreadsheet.

        Args:
            city: Destination city, such as Goa.
            nights: Number of nights to stay.
            budget: low, medium, or high.
        """
        print(f"TOOL find_hotels city={city} nights={nights} budget={budget}")
        stays = HOTELS.get(_city_key(city), [])
        if not stays:
            return {
                "city": city,
                "hotels": [],
                "note": f"No hotels in the catalog for {city}.",
                "available_cities": AVAILABLE_CITIES,
            }
        allowed = {budget.lower()}
        if budget.lower() == "medium":
            allowed.add("high")
        chosen = [hotel for hotel in stays if hotel["budget"] in allowed] or stays
        return {
            "city": city,
            "nights": nights,
            "hotels": [
                {
                    "name": hotel["name"],
                    "price_per_night_inr": hotel["price_per_night"],
                    "stay_total_inr": hotel["price_per_night"] * nights,
                    "rating": hotel["rating"],
                }
                for hotel in chosen
            ],
        }

    @function_tool
    async def find_activities(
        self,
        context: RunContext,
        city: str,
        interests: list[str],
        duration: int,
    ) -> dict:
        """Build a day-by-day activity plan from the workshop spreadsheet.

        Args:
            city: Destination city, such as Goa or Mussoorie.
            interests: Interests to prefer, such as beaches and nightlife.
            duration: Number of days to plan.
        """
        print(f"TOOL find_activities city={city} interests={interests} duration={duration}")
        catalog = ACTIVITIES.get(_city_key(city), [])
        if not catalog:
            return {
                "city": city,
                "days": [],
                "note": f"No activities in the catalog for {city}.",
                "available_cities": AVAILABLE_CITIES,
            }
        wanted = set(_normalize_interests(interests))
        days = []
        for plan in catalog:
            if plan["day"] > duration:
                continue
            if wanted and not (wanted & set(plan["interests"])):
                continue
            days.append({"day": plan["day"], "items": plan["items"]})
        if not days:
            days = [
                {"day": plan["day"], "items": plan["items"]}
                for plan in catalog
                if plan["day"] <= duration
            ]
        return {"city": city, "duration_days": duration, "days": days}


server = AgentServer(job_executor_type=JobExecutorType.THREAD)


@server.rtc_session()
async def entrypoint(ctx: JobContext):
    session = AgentSession(
        stt="deepgram/nova-3",
        llm="openai/gpt-4.1-mini",
        tts="cartesia/sonic-2:9626c31c-bec5-4cca-baa8-f8ba9e84c8bc",
        vad=silero.VAD.load(),
        turn_detection="vad",
    )
    await session.start(
        agent=TripMate(),
        room=ctx.room,
        room_input_options=RoomInputOptions(
            noise_cancellation=noise_cancellation.BVC(),
        ),
    )
    await ctx.connect()


await run_stage(server, "Stage 4 — three tools")


2026-10-21 13:51:51,571 INFO shutting down worker
2026-10-21 13:51:51,575 INFO starting worker
2026-10-21 13:51:51,576 INFO plugin registered
2026-10-21 13:51:51,577 INFO preloading plugins
2026-10-21 13:51:51,582 INFO HTTP server listening on :36565
2026-10-21 13:51:51,611 INFO registered worker


Closing the previous stage...
Stage 4 — three tools is starting.
Wait for 'registered worker', then connect on the website and say the Goa sentence.
Stop this cell before you run the next stage.



2026-10-21 13:51:57,086 INFO received job request
2026-10-21 13:51:57,091 WARNING no warmed process available for job, waiting for one to be created
2026-10-21 13:51:57,094 INFO initializing job runner
2026-10-21 13:51:57,096 INFO job runner initialized
2026-10-21 13:51:57,177 WARNING turn_detection is deprecated and will be removed in v2.0. Use turn_handling=TurnHandlingOptions(...) instead
2026-10-21 13:51:57,185 WARNING RoomInputOptions and RoomOutputOptions are deprecated, use RoomOptions instead
2026-10-21 13:51:57,195 INFO adaptive interruption is disabled by default in production mode
2026-10-21 13:51:57,240 INFO livekit_signaling::signal_stream:53:livekit_signaling::signal_stream - connecting to wss://test-agent-5f42hkx0.oashburn1b.production.livekit.cloud/rtc/v1
2026-10-21 13:51:57,297 INFO HTTP Request: GET https://agent-gateway.livekit.cloud/v1/models "HTTP/1.1 200 OK"
2026-10-21 13:51:57,755 INFO RoomIO linked to participant
2026-10-21 13:52:16,343 INFO HTTP Request: POST h

CancelledError: 

In [ ]:
from livekit.agents import (
    Agent,
    AgentServer,
    AgentSession,
    JobContext,
    JobExecutorType,
    RoomInputOptions,
    RunContext,
    TurnHandlingOptions,
    function_tool,
    inference,
)
from livekit.plugins import noise_cancellation, silero
from livekit.plugins.turn_detector.multilingual import MultilingualModel



def _normalize_interests(interests: list[str]) -> list[str]:
    cleaned = []
    for interest in interests:
        value = interest.strip().lower()
        if value in {"beach", "beachside"}:
            value = "beaches"
        if value in {"night life", "party", "parties", "clubs"}:
            value = "nightlife"
        if value in {"hill", "hills", "hill station"}:
            value = "mountains"
        if value:
            cleaned.append(value)
    return cleaned


class TripMate(Agent):
    def __init__(self) -> None:
        super().__init__(
            instructions=(
                "You are TripMate, a voice travel planner for trips in India. "
                "Speak in short spoken sentences. "
                "Never invent a city, hotel, price, or activity. "
                "Call search_destinations first when the traveler describes a trip. "
                "After you name a city, call find_hotels for that city, then call find_activities. "
                "For a medium budget, prefer the lower-priced well-rated hotel. "
                "If a tool says the city is not in the catalog, say that and name a city that is."
            ),
        )

    @function_tool
    async def search_destinations(
        self,
        context: RunContext,
        interests: list[str],
        duration: int,
        budget: str = "medium",
    ) -> dict:
        """Rank cities from the workshop spreadsheet for this trip.

        Args:
            interests: What the traveler wants, such as beaches or nightlife.
            duration: Trip length in days.
            budget: low, medium, or high.
        """
        print(f"TOOL search_destinations interests={interests} duration={duration} budget={budget}")
        wanted = set(_normalize_interests(interests))
        ranked = []
        for place in DESTINATIONS:
            overlap = wanted & set(place["interests"])
            if wanted and not overlap:
                continue
            ranked.append({
                "city": place["city"],
                "score": place["score"],
                "matched_interests": sorted(overlap),
                "why": place["why"],
                "duration_days": duration,
                "budget": budget,
            })
        ranked.sort(key=lambda item: item["score"], reverse=True)
        return {"destinations": ranked[:3], "available_cities": AVAILABLE_CITIES}

    @function_tool
    async def find_hotels(
        self,
        context: RunContext,
        city: str,
        nights: int,
        budget: str = "medium",
    ) -> dict:
        """Find hotels in a city from the workshop spreadsheet.

        Args:
            city: Destination city, such as Goa.
            nights: Number of nights to stay.
            budget: low, medium, or high.
        """
        print(f"TOOL find_hotels city={city} nights={nights} budget={budget}")
        stays = HOTELS.get(_city_key(city), [])
        if not stays:
            return {
                "city": city,
                "hotels": [],
                "note": f"No hotels in the catalog for {city}.",
                "available_cities": AVAILABLE_CITIES,
            }
        allowed = {budget.lower()}
        if budget.lower() == "medium":
            allowed.add("high")
        chosen = [hotel for hotel in stays if hotel["budget"] in allowed] or stays
        return {
            "city": city,
            "nights": nights,
            "hotels": [
                {
                    "name": hotel["name"],
                    "price_per_night_inr": hotel["price_per_night"],
                    "stay_total_inr": hotel["price_per_night"] * nights,
                    "rating": hotel["rating"],
                }
                for hotel in chosen
            ],
        }

    @function_tool
    async def find_activities(
        self,
        context: RunContext,
        city: str,
        interests: list[str],
        duration: int,
    ) -> dict:
        """Build a day-by-day activity plan from the workshop spreadsheet.

        Args:
            city: Destination city, such as Goa or Mussoorie.
            interests: Interests to prefer, such as beaches and nightlife.
            duration: Number of days to plan.
        """
        print(f"TOOL find_activities city={city} interests={interests} duration={duration}")
        catalog = ACTIVITIES.get(_city_key(city), [])
        if not catalog:
            return {
                "city": city,
                "days": [],
                "note": f"No activities in the catalog for {city}.",
                "available_cities": AVAILABLE_CITIES,
            }
        wanted = set(_normalize_interests(interests))
        days = []
        for plan in catalog:
            if plan["day"] > duration:
                continue
            if wanted and not (wanted & set(plan["interests"])):
                continue
            days.append({"day": plan["day"], "items": plan["items"]})
        if not days:
            days = [
                {"day": plan["day"], "items": plan["items"]}
                for plan in catalog
                if plan["day"] <= duration
            ]
        return {"city": city, "duration_days": duration, "days": days}


server = AgentServer(job_executor_type=JobExecutorType.THREAD)


@server.rtc_session()
async def entrypoint(ctx: JobContext):
    session = AgentSession(
        stt="deepgram/nova-3",
        llm="openai/gpt-4.1-mini",
        tts="cartesia/sonic-2:9626c31c-bec5-4cca-baa8-f8ba9e84c8bc",
        vad=silero.VAD.load(),
        turn_handling=TurnHandlingOptions(
            turn_detection=inference.TurnDetector(),
            mode="fixed",
            min_delay=4,
            max_delay=6.0,
        ),
    )
    await session.start(
        agent=TripMate(),
        room=ctx.room,
        room_input_options=RoomInputOptions(
            noise_cancellation=noise_cancellation.BVC(),
        ),
    )
    await ctx.connect()


await run_stage(server, "Stage 5 — turn detection")


2026-10-21 13:59:04,046 INFO shutting down worker
2026-10-21 13:59:04,053 INFO process exiting
2026-10-21 13:59:04,212 INFO starting worker
2026-10-21 13:59:04,213 INFO plugin registered
2026-10-21 13:59:04,214 INFO plugin registered
2026-10-21 13:59:04,215 INFO plugin registered
2026-10-21 13:59:04,217 INFO plugin registered
2026-10-21 13:59:04,218 INFO preloading plugins
2026-10-21 13:59:04,219 INFO starting inference executor


Closing the previous stage...
Stage 4 — three tools is starting.
Wait for 'registered worker', then connect on the website and say the Goa sentence.
Stop this cell before you run the next stage.



2026-10-21 13:59:04,242 INFO initializing process
2026-10-21 13:59:18,862 INFO process initialized
2026-10-21 13:59:18,872 INFO HTTP server listening on :37187
2026-10-21 13:59:18,927 INFO registered worker
2026-10-21 13:59:38,352 INFO received job request
2026-10-21 13:59:38,357 WARNING no warmed process available for job, waiting for one to be created
2026-10-21 13:59:38,360 INFO initializing job runner
2026-10-21 13:59:38,722 INFO job runner initialized
2026-10-21 13:59:38,804 WARNING RoomInputOptions and RoomOutputOptions are deprecated, use RoomOptions instead
2026-10-21 13:59:38,814 INFO adaptive interruption is disabled by default in production mode
2026-10-21 13:59:38,860 INFO livekit_signaling::signal_stream:53:livekit_signaling::signal_stream - connecting to wss://test-agent-5f42hkx0.oashburn1b.production.livekit.cloud/rtc/v1
2026-10-21 13:59:38,919 INFO HTTP Request: GET https://agent-gateway.livekit.cloud/v1/models "HTTP/1.1 200 OK"
2026-10-21 13:59:39,400 INFO RoomIO linke

TOOL search_destinations interests=['go out'] duration=2 budget=medium


2026-10-21 13:59:48,565 INFO HTTP Request: POST https://agent-gateway.livekit.cloud/v1/chat/completions "HTTP/1.1 200 OK"
2026-10-21 13:59:52,983 INFO HTTP Request: POST https://agent-gateway.livekit.cloud/v1/chat/completions "HTTP/1.1 200 OK"
2026-10-21 13:59:55,361 INFO HTTP Request: POST https://agent-gateway.livekit.cloud/v1/chat/completions "HTTP/1.1 200 OK"


TOOL search_destinations interests=['nightlife', 'beaches'] duration=2 budget=medium


2026-10-21 13:59:57,159 INFO HTTP Request: POST https://agent-gateway.livekit.cloud/v1/chat/completions "HTTP/1.1 200 OK"


TOOL find_hotels city=Goa nights=2 budget=medium
TOOL find_activities city=Goa interests=['nightlife', 'beaches'] duration=2


2026-10-21 13:59:57,676 INFO HTTP Request: POST https://agent-gateway.livekit.cloud/v1/chat/completions "HTTP/1.1 200 OK"
2026-10-21 14:00:07,157 INFO closing agent session due to participant disconnect (disable via `RoomInputOptions.close_on_disconnect=False`)


CancelledError: 

# Stage 5 — conversational agent


Stop stage 4 first.

**Change:** one argument on `AgentSession`, `turn_handling`. The tools and the instructions are the same as stage 4.

Until now the agent used a simple voice-activity detector: it heard silence and assumed you were done. A short pause in the middle of a sentence was enough to make it reply.

`MultilingualModel` is a turn detector. It looks at the words so far and decides whether the turn is finished. The files for that model were downloaded in the setup cell.

**Say the same Goa sentence, but pause** after "Plan a 4-day Goa trip..." for about two seconds, then finish with "with beaches and nightlife, medium budget."

**Listen for:** stage 4 would have started answering at the pause. Stage 5 should wait, then answer with the same sheet-backed city, hotel, and activities. You should still see all three `TOOL` lines, and only after you finish the sentence.

That is the whole arc:

- Stage 1 was a voice chatbot.
- Stage 3 was a tool-using agent.
- Stage 5 is a conversational agent: same tools, plus a sense of when you are finished talking.


In [ ]:
import os

from livekit.agents import (
    Agent,
    AgentServer,
    AgentSession,
    JobContext,
    JobExecutorType,
    RoomInputOptions,
    RunContext,
    TurnHandlingOptions,
    function_tool,
    inference,
)
from livekit.plugins import noise_cancellation, silero
from livekit.plugins.turn_detector.multilingual import MultilingualModel

# We import the simli plugin at the top level of the cell so it is registered on the main thread
try:
    from livekit.plugins import simli as simli_plugin
except ImportError:
    simli_plugin = None


class TripMate(Agent):
    def __init__(self) -> None:
        super().__init__(
            instructions=(
                "You are TripMate, a voice travel planner for trips in India. "
                "Speak in short spoken sentences. "
                "Never invent a city, hotel, price, or activity. "
                "Call search_destinations first when the traveler describes a trip. "
                "After you name a city, call find_hotels for that city, then call find_activities. "
                "For a medium budget, prefer the lower-priced well-rated hotel. "
                "If a tool says the city is not in the catalog, say that and name a city that is."
            ),
        )

    @function_tool
    async def search_destinations(
        self,
        context: RunContext,
        interests: list[str],
        duration: int,
        budget: str = "medium",
    ) -> dict:
        """Rank cities from the workshop spreadsheet for this trip.

        Args:
            interests: What the traveler wants, such as beaches or nightlife.
            duration: Trip length in days.
            budget: low, medium, or high.
        """
        print(f"TOOL search_destinations interests={interests} duration={duration} budget={budget}")
        wanted = set(_normalize_interests(interests))
        ranked = []
        for place in DESTINATIONS:
            overlap = wanted & set(place["interests"])
            if wanted and not overlap:
                continue
            ranked.append({
                "city": place["city"],
                "score": place["score"],
                "matched_interests": sorted(overlap),
                "why": place["why"],
                "duration_days": duration,
                "budget": budget,
            })
        ranked.sort(key=lambda item: item["score"], reverse=True)
        return {"destinations": ranked[:3], "available_cities": AVAILABLE_CITIES}

    @function_tool
    async def find_hotels(
        self,
        context: RunContext,
        city: str,
        nights: int,
        budget: str = "medium",
    ) -> dict:
        """Find hotels in a city from the workshop spreadsheet.

        Args:
            city: Destination city, such as Goa.
            nights: Number of nights to stay.
            budget: low, medium, or high.
        """
        print(f"TOOL find_hotels city={city} nights={nights} budget={budget}")
        stays = HOTELS.get(_city_key(city), [])
        if not stays:
            return {
                "city": city,
                "hotels": [],
                "note": f"No hotels in the catalog for {city}.",
                "available_cities": AVAILABLE_CITIES,
            }
        allowed = {budget.lower()}
        if budget.lower() == "medium":
            allowed.add("high")
        chosen = [hotel for hotel in stays if hotel["budget"] in allowed] or stays
        return {
            "city": city,
            "nights": nights,
            "hotels": [
                {
                    "name": hotel["name"],
                    "price_per_night_inr": hotel["price_per_night"],
                    "stay_total_inr": hotel["price_per_night"] * nights,
                    "rating": hotel["rating"],
                }
                for hotel in chosen
            ],
        }

    @function_tool
    async def find_activities(
        self,
        context: RunContext,
        city: str,
        interests: list[str],
        duration: int,
    ) -> dict:
        """Build a day-by-day activity plan from the workshop spreadsheet.

        Args:
            city: Destination city, such as Goa or Mussoorie.
            interests: Interests to prefer, such as beaches and nightlife.
            duration: Number of days to plan.
        """
        print(f"TOOL find_activities city={city} interests={interests} duration={duration}")
        catalog = ACTIVITIES.get(_city_key(city), [])
        if not catalog:
            return {
                "city": city,
                "days": [],
                "note": f"No activities in the catalog for {city}.",
                "available_cities": AVAILABLE_CITIES,
            }
        wanted = set(_normalize_interests(interests))
        days = []
        for plan in catalog:
            if plan["day"] > duration:
                continue
            if wanted and not (wanted & set(plan["interests"])):
                continue
            days.append({"day": plan["day"], "items": plan["items"]})
        if not days:
            days = [
                {"day": plan["day"], "items": plan["items"]}
                for plan in catalog
                if plan["day"] <= duration
            ]
        return {"city": city, "duration_days": duration, "days": days}


server = AgentServer(job_executor_type=JobExecutorType.THREAD)


@server.rtc_session()
async def entrypoint(ctx: JobContext):
    session = AgentSession(
        stt="deepgram/nova-3",
        llm="openai/gpt-4.1-mini",
        tts="cartesia/sonic-2:9626c31c-bec5-4cca-baa8-f8ba9e84c8bc",
        vad=silero.VAD.load(),
        turn_handling=TurnHandlingOptions(
            turn_detection=inference.TurnDetector(),
            mode="fixed",
            min_delay=4,
            max_delay=6.0,
        ),
    )

    agent = TripMate()

    # If the bonus Simli cell was run, attach an avatar
    simli_key = os.environ.get("SIMLI_API_KEY", "").strip()
    if simli_key and simli_plugin:
        face_id = os.environ.get("SIMLI_FACE_ID", "cace3ef7-a4c4-425d-a8cf-a5358eb0c427")
        avatar = simli_plugin.AvatarSession(
        simli_config=simli_plugin.SimliConfig(api_key=simli_key, face_id=face_id)
    )
    print(f"Simli avatar ON — face {face_id}")
    await avatar.start(session, room=ctx.room)
    await avatar.wait_for_join()

    await session.start(
        agent=agent,
        room=ctx.room,
        room_input_options=RoomInputOptions(
            noise_cancellation=noise_cancellation.BVC(),
        ),
    )
    await ctx.connect()


await run_stage(server, "Stage 5 — conversational agent")

/tmp/ipykernel_3067/3819901289.py:16: DeprecationWarning: `livekit.plugins.turn_detector` is deprecated and will be removed in a future release. Use `livekit.agents.inference.TurnDetector` instead.
  from livekit.plugins.turn_detector.multilingual import MultilingualModel
2026-10-21 13:54:01,081 INFO shutting down worker
2026-10-21 13:54:01,086 INFO starting worker
2026-10-21 13:54:01,087 INFO plugin registered
2026-10-21 13:54:01,090 INFO plugin registered
2026-10-21 13:54:01,091 INFO plugin registered
2026-10-21 13:54:01,093 INFO plugin registered
2026-10-21 13:54:01,094 INFO preloading plugins
2026-10-21 13:54:01,095 INFO starting inference executor


Closing the previous stage...
Stage 5 — conversational agent is starting.
Wait for 'registered worker', then connect on the website and say the Goa sentence.
Stop this cell before you run the next stage.



2026-10-21 13:54:03,855 INFO initializing process
2026-10-21 13:54:16,858 INFO process initialized
2026-10-21 13:54:16,863 INFO HTTP server listening on :42545
2026-10-21 13:54:16,894 INFO registered worker
2026-10-21 13:54:25,809 INFO received job request
2026-10-21 13:54:25,813 WARNING no warmed process available for job, waiting for one to be created
2026-10-21 13:54:25,816 INFO initializing job runner
2026-10-21 13:54:25,818 INFO job runner initialized
2026-10-21 13:54:25,902 ERROR unhandled exception while running the job task
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/livekit/agents/ipc/job_proc_lazy_main.py", line 349, in _traceable_entrypoint
    await self._job_entrypoint_fnc(job_ctx)
  File "/tmp/ipykernel_3067/3819901289.py", line 182, in entrypoint
    print(f"Simli avatar ON — face {face_id}")
                                    ^^^^^^^
UnboundLocalError: cannot access local variable 'face_id' where it is not associated with a value


CancelledError: 

# Bonus — Simli avatar (optional)

This cell is **optional**. If you run it, stage 5 will show a Simli avatar on the website. If you skip it, nothing breaks — you just see the plane icon.

### Setup

1. Add a Colab Secret called `SIMLI_API_KEY` with your Simli API key (22 characters). Turn **Notebook access** on.
2. Optionally add `SIMLI_FACE_ID` if you want a different face. The default is a young woman.
3. Run this cell, then **re-run Stage 5**. The avatar appears on the website.

The API key is **not** the face ID. The key is shorter (22 characters). The face ID is a UUID (36 characters).


In [ ]:
import os

try:
    from google.colab import userdata
    simli_key = userdata.get("SIMLI_API_KEY")
except Exception:
    simli_key = None

if simli_key:
    simli_key = simli_key.strip().strip('"').strip("'")
    os.environ["SIMLI_API_KEY"] = simli_key

    # Optional face ID override
    try:
        face_id = userdata.get("SIMLI_FACE_ID")
        if face_id:
            os.environ["SIMLI_FACE_ID"] = face_id.strip().strip('"').strip("'")
    except Exception:
        pass

    face = os.environ.get("SIMLI_FACE_ID", "cace3ef7-a4c4-425d-a8cf-a5358eb0c427")
    print(f"✅ Simli avatar enabled — face {face}")
    print(f"   SIMLI_API_KEY length: {len(simli_key)}")
    if len(simli_key) > 30:
        print("   ⚠️  That key looks too long. The API key is ~22 characters. The face ID is 36. Make sure you pasted the API key, not the face ID.")
    print("\n Now re-run Stage 5. The avatar will appear on the website.")
else:
    print("ℹ️  No SIMLI_API_KEY found in Colab Secrets. Skipping avatar.")
    print("   All stages still work — you just won't see a face.")


✅ Simli avatar enabled — face cace3ef7-a4c4-425d-a8cf-a5358eb0c427
   SIMLI_API_KEY length: 22

 Now re-run Stage 5. The avatar will appear on the website.


## What you just walked


Same website. Same Goa sentence. Five agents.

1. **Voice chatbot** — it could talk, and it invented the price.
2. **Instructions** — the name changed. The facts did not.
3. **One tool** — `find_hotels` read the spreadsheet. The price became Sea Breeze, ₹2700.
4. **All three tools** — `search_destinations`, then `find_hotels`, then `find_activities`. The model followed an order and planned the whole trip from the sheet.
5. **A conversation** — the turn detector waited out a pause. The tools did not change.

The budget tool and the Simli face are the same kind of step as stage 3: another tool, or another output. They are not a new kind of agent.

---
